# $0.2 + 0.1 \ne 0.3$ 🤯😵‍💫
### (floating point arithmetic)

### Intended Learning Outcomes (ILOs)

By the end of this lecture, you should be able to:
1. Represent floating point numbers using IEEE-754 standard.
2. Identify common issues with floating point arithmetic, such as rounding errors, overflow, and underflow.
3. Recognise some limitations of floating point arithmetic and how they can impact numerical accuracy.
4. Apply simple techniques to reduce floating-point errors (summation order, reformulation, rescaling, relative tolerances).

## Surprises 🫢😯🤯

### Surprise 1

#### Maths says...
For any $x \in \mathbb{R}$ if $x \ne 0$
$$1 + x \ne 1$$

#### Computer says...
$$x = 10^{-16}$$

In [ ]:
x = 1e-16  # this is 10^{-16}

print(1 + x != 1)

### Surprise 2

#### Maths says...
$$\frac{1}{10} + \frac{2}{10} = \frac{3}{10}$$

$$0.1 + 0.2 = 0.3$$

#### Computer says...

In [ ]:
print(0.1 + 0.2 == 0.3)

### Surprise 3

#### Maths says...
$$(a + b) + c = a + (b + c)$$

#### Computer says...

In [ ]:
a = 1e16
b = -1e16
c = 1.0

print('Associativity works:', (a + b) + c == a + (b + c))

In [ ]:
print("(a + b) + c = ", (a + b) + c)
print("a + (b + c) = ", a + (b + c))

### Surprise 4

#### Maths says...
$$1.000000\mathbf{1} - 1.0000000 = 0.0000001 = 10^{-7}$$

#### Computer says...

In [1]:
a = 1.0000001
b = 1.0000000

print("a - b == 1e-7:", a - b == 1e-7)

a - b == 1e-7: False


In [2]:
print("a - b =", a - b)

a - b = 1.0000000005838672e-07


### Surprise 5

#### Maths says...
$$a \times b = \underbrace{b + b + ... + b}_{\text{a times}}$$

#### Computer says...

In [3]:
a = 100
b = 0.1

s = 0
for _ in range(a):
    s += b

print("a * b == b + b + b +...:", a * b == s)

a * b == b + b + b +...: False


In [4]:
print("a * b =", a * b)
print("b + b + b + b + ...=", s)

a * b = 10.0
b + b + b + b + ...= 9.99999999999998


### Maths says yes, but...

<div>
<center>
    <img src="images/computer-says-no.jpeg" width="700" alt="Computer says no" />
</center>
</div>

[Little Britain: https://www.youtube.com/watch?v=0n_Ty_72Qds]

### Floating point number (in decimal representation)

$$x = \underbrace{28\,763}_\text{integer part}.\overbrace{437}^\text{fractional part}$$

- Integer part:

$$
\begin{split}
28\,763 & = 20\,000 + 8\,000 + 700 + 60 + 3 \\
        & = 2 \times 10\,000 + 8 \times 1\,000 + 7 \times 100 + 6 \times 10 + 3 \times 1 \\
        & = \underbrace{2 \times 10^{4}} + \underbrace{8 \times 10^{3}} + \underbrace{7 \times 10^{2}} + \underbrace{6 \times 10^{1}} + \underbrace{3 \times 10^{0}}
\end{split}
$$

- Fractional part:

$$
\begin{split}
0.437 & = \frac{4}{10} + \frac{3}{100} + \frac{7}{1\,000}\\
      & = 4 \times \frac{1}{10} + 3 \times \frac{1}{100} + 7 \times \frac{1}{1\,000}\\
      & = \underbrace{4 \times 10^{-1}} + \underbrace{3 \times 10^{-2}} + \underbrace{7 \times 10^{-3}}
\end{split}
$$

### Floating point number (in decimal representation)

$$x = \underbrace{28\,763}_\text{integer part}.\overbrace{437}^\text{fractional part}$$

$$
\underbrace{a_{n}a_{n-1}\ldots a_1 a_0}_{\text{integer part}} \,\,.\,\, \overbrace{b_{1} b_{2} b_{3} \ldots}^{\text{fractional part}} = \underbrace{\sum_{i=0}^{n} a_{i} \times 10^{i}}_{\text{integer part}} + \overbrace{\sum_{i=1}^{\infty} b_{i} \times 10^{-i}}^{\text{fractional part}}
$$

- Digits in base-10: $a_{i}, b_{i} \in \{0, 1, 2, \ldots, 9\}$
- Integer part has a finite number of digits.
- Fractional part can have:
  - **finite** number of digits - some rational numbers, e.g.
  $$\frac{1}{4} = 0.25$$
  - **infinite** number of digits - all irrational numbers ($\pi$, $e$...) and some rational numbers, e.g.
  $$\frac{1}{3}=0.333333333333....$$

### Floating point number (in binary representation)

$$x = (\underbrace{1\,011}_{\text{integer part}}.\overbrace{101}^{\text{fractional part}})_{2}$$

- **Note:** We write $(1\,011.101)_{2}$ ($2$ in subscript) to prevent us from interpreting the number as decimal (*one thousand eleven point one zero one*)

- Integer part:
$$
\begin{split}
(1\,011)_{2} & = \overbrace{1 \times 2^{3}} + \overbrace{0 \times 2^{2}} + \overbrace{1 \times 2^{1}} + \overbrace{1 \times 2^{0}}\\
             & = 1 \times 8 + 0 \times 4 + 1 \times 2 + 1 \times 1\\
             & = 8 + 0 + 2 + 1\\
             & = (11)_{10}
\end{split}
$$

- Fractional part:
$$
\begin{split}
(0.101)_{2} & = 1 \times 2^{-1} + 0 \times 2^{-2} + 1 \times 2^{-3}\\
            & = 1 \times \frac{1}{2} + 0 \times \frac{1}{4} + 1 \times \frac{1}{8}\\
            & = 0.5 + 0 + 0.125\\
            & = (0.625)_{10}
\end{split}
$$

### Floating point number (in binary representation)

$$
\left(\underbrace{a_{n} a_{n-1}\ldots a_1 a_0}_{\text{integer part}} \,\, . \,\, \overbrace{\, b_{1} b_{2} b_{3} \ldots}^{\text{fractional part}}\right)_{2} = \underbrace{\sum_{i=0}^{n} a_{i} \times 2^{i}}_{\text{integer part}} + \overbrace{\sum_{i=1}^{\infty} b_{i} \times 2^{-i}}^{\text{fractional part}}
$$

- Digits in base-2: $a_{i}, b_{i} \in \{0, 1\}$
- Integer part has a finite number of digits. Fractional part can have finite or infinite number of digits.
- In integer part, we call:
  - $a_{0}$: Least Significant Bit (LSB)
  - $a_{n}$: Most Significant Bit (MSB)
- **Binary** number, but we still call it a **decimal point**? If you're picky, call it a **radix point**.

### Floating point number (in base-$\beta$ representation)

$$
\left(\underbrace{a_{n} a_{n-1}\ldots a_1 a_0}_{\text{integer part}} \,\, . \,\, \overbrace{\, b_{1} b_{2} b_{3} \ldots}^{\text{fractional part}}\right)_{\beta} = \underbrace{\sum_{i=0}^{n} a_{i} \times \beta^{i}}_{\text{integer part}} + \overbrace{\sum_{i=1}^{\infty} b_{i} \times \beta^{-i}}^{\text{fractional part}}
$$

| base ($\beta$) | name | digits |
| - | - | - |
| 2 | binary | 0, 1 |
| 8 | octal | 0, 1, 2, 3, 4, 5, 6, 7 |
| 10 | decimal | 0, 1, 2, 3, 4, 5, 6, 7, 8, 9 |
| 16 | hexadecimal | 0, 1, 2, ..., 9, a, b, c, d, e, f |

## Decimal to binary conversion

### Step 1: Convert the integer part

1. **Divide the integer part** of the decimal number by 2.
2. **Record the remainder** (0 or 1) as the **least significant bit (LSB)**.
3. **Repeat the process**: divide the quotient from the previous step by 2 and record the remainder until the quotient becomes 0.
4. **The binary representation** of the integer is the remainders read from **bottom to top** - from MSB to LSB.

#### **Example: Convert $(11)_{10}$ to binary**

| calculation       | quotient | remainder |
| ----------------- | -------- | --------- |
| $11 \div 2$      | $5$     | $1$ (LSB)      |
| $5 \div 2$       | $2$     | $1$       |
| $2 \div 2$       | $1$     | $0$       |
| $1 \div 2$       | $0$ (stop)     | $1$ (MSB)       |

- We read the remainders from bottom to top: $1$ (MSB), $0$, $1$, $1$ (LSB):


$$(11)_{10} = (1011)_2$$

### Step 2: Convert the fractional part

1. **Multiply the fractional part** of the decimal number by 2.
2. Record the **integer part** (0 or 1) of the result as the **next bit** in the binary representation.
3. **Repeat the process**: multiply the fractional part of the result by 2 and record the integer part.
4. **Stop** when the fractional part becomes 0 (or continue for a certain number of bits if it does not terminate).

#### **Example: Convert $(0.625)_{10}$ to binary**

| calculation         | result         | integer part | fractional part |
| ------------------- | -------------- | ------------ | ------------------------- |
| $0.625 \times 2$    | $1.25$        | $1$          | $0.25$                   |
| $0.25 \times 2$     | $0.50$        | $0$          | $0.50$                   |
| $0.5 \times 2$      | $1.0$         | $1$          | $0.0$                    |

Thus, 0.625 in binary, when we read the integer parts from top to bottom:
$$(0.625)_{10} = (0.101)_{2}$$

$$(11)_{10} = (1011)_2$$

$$+$$

$$(0.625)_{10} = (0.101)_{2}$$

$$\downarrow$$

$$(11.625)_{10} = (1011.101)_{2}$$

### General base conversion

#### Integer part

- We can write an integer $x$ in the nested polynomial form:

$$
\begin{split}
x &= (c_{n}c_{n-1} \ldots c_{1}c_{0})_{\beta} \\
  &= \sum_{i=0}^{n} c_{i} \times \beta^{i} \\
  &= c_{0} + \beta(c_{1} + \beta (c_{2} + \beta (\ldots)))
\end{split}
$$

- Dividing $x$ by $\beta$ results in remainder $c_{0}$ and quotient $c_{1} + \beta (c_{2} + \beta (\ldots))$.
- By repeating the division we obtain digits $c_{0}$, $c_{1}$, $c_{2}$, etc.

#### Fractional part

- A real number $x < 1$ in base-$\beta$ can be written as:
$$
\begin{split}
x &= (0.c_{1}c_{2}c_{3}\ldots)_{\beta} \\
  &= \sum_{i=1}^{\infty}c_{i}\beta^{-i}
\end{split}
$$

- If we multiply $x$ by $\beta$, we get:
$$
\beta x = (c_{1}.c_{2}c_{3}\ldots)_{\beta}
$$

- Digit $c_{1}$ we get by taking the integer part of $\beta x$.
- By repeating the multiplication by $\beta$ we obtain all digits (if the number has a finite representation) or until we obtain enough significant digits.

### Scientific notation

- Instead of splitting a float into the integer and fractional parts, scientific notation is a way of representing a number in the form:

$$x = \pm \,\text{mantissa} \times 10^{\text{exponent}}$$

$$
\begin{split}
37541.23 &= +\overbrace{37.54123}^{\text{mantissa}} \times 10^{\overbrace{3}^{\text{exponent}}}\\ 
         &= +3.754123 \times 10^{4}\\
         &= +0.3754123 \times 10^{5}
\end{split}
$$

The scientific notation is **not unique**.

### Normalised scientific notation

- To ensure uniqueness, we define **normalised** scientific notation.

$$x = \pm\overbrace{d_{1}.d_{2}d_{3}\ldots}^{\text{mantissa}\,\,m} \times 10^{e}, \quad d_{1} \ne 0$$

- The notation is normalised if mantissa $m = d_{1}.d_{2}d_{3}\ldots$ is $1 \le m < 10$.
- We keep moving the decimal point until we have only one non-zero digit in the integer part.

$$
\begin{split}
37541.23 &= \overbrace{+3.754123 \times 10^{4}}^{\text{normalised}}\\ 
         &= \underbrace{+0.3754123 \times 10^{5}}_{\text{not normalised}}
\end{split}
$$

### Normalised scientific notation (in decimal form)

Any $x \in \mathbb{R}$ if $x \ne 0$ can be written as:

$$
x = \pm m \times 10^{e}, \quad 1 \le m < 10,\quad e \in \mathbb{Z}
$$

| label | name |
| - | - |
| $\pm$ | sign
| $m$ | mantissa |
| $e$ | exponent |

### Normalised scientific notation (in binary form)

Any $x \in \mathbb{R}$ if $x \ne 0$ can be written as:

$$
\begin{split}
x &= \pm \overbrace{b_{0}.b_{1}b_{2}\ldots}^{\text{mantissa}\,\,m} \times 2^{e}\\
  &= \pm 1.b_{1}b_{2}\ldots \times 2^{e}\\
  &= \pm m \times 2^{e}, \quad 1 \le m < 2
\end{split}
$$

- **Note**: Since $b_{0}$ can only be $1$, we write $m = 1.b_{1}b_{2}\ldots$.

| label | name |
| - | - |
| $\pm$ | sign
| $m$ | mantissa |
| $e$ | exponent |

## IEEE-754

- Earlier, computer manufacturers developed their own floating-point number representations.
    - There were inconsistencies in results between different machines.
    - The result of $a + b$ was different on different machines. 🤯😵‍💫
- In 1985, IEEE (Institute of Electrical and Electronics Engineers) established the floating-point standard IEEE-754.

### To save a float, we need...

Normalised scientific notation:

$$
\begin{split}
x &= \pm \overbrace{1.b_{1}b_{2}\ldots}^{\text{mantissa}\,\,m} \times 2^{e}\\
  &= (-1)^{s} \times 1.b_{1}b_{2}\ldots \times 2^{e}\\
\end{split}
$$

Practically, to save a float, we have to save three integers:

| label | name |
| - | - |
| $s$ | sign |
| $f$ | fraction bits $b_{1}b_{2}\ldots$ of the mantissa (the leading $1$ is not stored - "hidden bit") |
| $e$ | exponent |

Let's say we find the following array of bits under the microscope:

$$\left(\overbrace{1}^{\text{sign}} \overbrace{10000000011}^{\text{exponent}}  \overbrace{1000111111001100101000101101101101100001101110110000}^{\text{mantissa}}\right)_{2} = (????)_{10}$$

### IEEE-754 levels of precision

| precision   | bits | sign | exponent | mantissa |
| ----------- | ---- | ---- | -------- | -------- |
| single      | 32   | 1    | 8        | 23       |
| double      | 64   | 1    | 11       | 52       |
| quadruple   | 128  | 1    | 15       | 112      |

**Do not memorise!** We do not have to know all the numbers, but we need to be aware of the basics.

### Single precision (32 bit)

According to IEEE-754, a single precision float is represented as:

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}, \quad 1.f = 1.b_{1}b_{2}b_{3}...$$

We need to save three integers: $s$, $f$, and $e$.

To keep things shorter, we write $1.f$ instead of $1.b_{1}b_{2}...$. Therefore, think about $f$ as an array of 23 bits.

### Single precision (32 bit) - in memory

<center>
    <img src="images/single-precision-layout.png" alt="Single precision bit layout" width="1300"/>
</center>

### Single precision (32 bit) - sign bit

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$

- 1 bit for the sign $s$
    - $s=0$ ($x > 0$)
    $$(-1)^0 = 1$$
    - $s=1$ ($x < 0$)
    $$(-1)^1 = -1$$ 

### Single precision (32 bit) - exponent

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$

- 8 bits for the exponent $e$
    - $(0)_{10} = (00\,000\,000)_{2} < e < (11\,111\,111)_{2} = (255)_{10}$
    - values $0$ and $255$ are reserved: $e = 0$ for $\pm 0$ and subnormal numbers, $e = 255$ for $\pm \infty$ and NaN (not a number).
    - Instead of using an extra bit for the sign of the exponent, IEEE-754 shifts $e$ by 127 to allow negative exponents (small $x$).
    $$-126 \le e-127 \le 127$$

### Why $e - 127$? (biased exponent)

- Small numbers need **negative** exponents, e.g. $0.15625 = (1.01)_{2} \times 2^{-3}$.
- Without a trick, we would also have to store the **sign of the exponent**:
    - an extra bit,
    - two zeros ($+0$ and $-0$ exponent),
    - extra logic to compare exponents.
- IEEE-754 instead stores $e$ as an **unsigned integer** ($0 \ldots 255$, always $\ge 0$) and subtracts a fixed **bias** $127$.

| stored bits | $e$ | real exponent $e - 127$ |
| - | - | - |
| `00000001` | $1$ | $-126$ (smallest) |
| `01010111` | $87$ | $-40$ (Worked example 2) |
| `01111111` | $127$ | $0$ |
| `10000011` | $131$ | $4$ (Worked example 1) |
| `11111110` | $254$ | $127$ (largest) |

- Bias is $2^{k-1} - 1$ for a $k$-bit exponent: single $2^{7} - 1 = 127$, double $2^{10} - 1 = 1023$ ($-1022 \le e - 1023 \le 1023$).
- Bonus: positive floats compare in the same order as their bits read as integers.

### Single precision (32 bit) - mantissa

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$    
    
- 23 bits for the fraction $f$ of the mantissa (aka significand) $m = 1.f$
  - $m = 1.f = 1.b_{1}b_{2}b_{3}...b_{23}$
  - Mantissa is limited by:
    $$(1.00000000000000000000000)_{2} = 1 \le 1.f \le (1.11111111111111111111111)_{2} = 2 - 2^{-23}$$


### What is the largest number?

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$   

- $s = 0$ (positive)
- $m = 1.f = (1.11111111111111111111111)_{2} = 2 - 2^{-23}$
- $e = (11111110)_{2} = 254$ ($255$ is reserved for infinity and NaN)

$$(-1)^{0} \times (2 - 2^{-23}) \times 2^{254-127} \approx 2^{128} \approx 3.4 \times 10^{38}$$

### What is the smallest positive (normal) number?

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$   

- $s = 0$ (positive)
- $m = 1.f = (1.00000000000000000000000)_{2} = 1$
- $e = (00000001)_{2} = 1$ ($0$ is reserved for zero and subnormal numbers)

$$(-1)^{0} \times 1 \times 2^{1-127} = 2^{-126} \approx 1.18 \times 10^{-38}$$

- **Note:** With $e = 0$, IEEE-754 also stores *subnormal* numbers (without the hidden $1$), down to $2^{-149} \approx 1.4 \times 10^{-45}$.

### For example...

$$\left(\overbrace{1}^{\text{sign}} \overbrace{10000001}^{\text{exponent}}  \overbrace{10110000000000000000000}^{\text{mantissa}}\right)_{2} = -(6.75)_{10}$$

- sign: $s=1$
- exponent: $e = (10000001)_{2} = (129)_{10}$
- mantissa: $m = (1.\mathbf{10110000000000000000000})_{2}$

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$

### Worked example 1: decimal → single precision

What are the 32 bits of $x = -24.98746$ in single precision?

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$

1. Sign $s$
2. Integer part $24$ → binary
3. Fractional part $0.98746$ → binary
4. Normalise and round → $f$
5. Exponent $e$
6. Assemble the 32 bits

`s | eeeeeeee | fffffffffffffffffffffff`

### Worked example 1 - Step 1: sign

$$x = -24.98746 < 0$$

$$s = 1$$

`1 | ???????? | ???????????????????????`

### Worked example 1 - Step 2: integer part

Convert $(24)_{10}$ by repeated division by 2:

| calculation | quotient | remainder |
| - | - | - |
| $24 \div 2$ | $12$ | $0$ (LSB) |
| $12 \div 2$ | $6$ | $0$ |
| $6 \div 2$ | $3$ | $0$ |
| $3 \div 2$ | $1$ | $1$ |
| $1 \div 2$ | $0$ (stop) | $1$ (MSB) |

Reading the remainders from bottom to top:

$$(24)_{10} = (11000)_{2}$$

### Worked example 1 - Step 3: how many fractional bits do we need?

$$24.98746 = (11000.\,?????\ldots)_{2} = (1.\underbrace{1000}_{4\text{ bits}}\,?????\ldots)_{2} \times 2^{4}$$

- $f$ has $23$ bits - $4$ of them come from the integer part.

- We need $23 - 4 = 19$ fractional bits...
- ...plus the $20^{\text{th}}$ bit to decide how to round.

### Worked example 1 - Step 3: fractional part (bits 1-10)

Multiply by 2, record the integer part, continue with the fractional part:

| bit | fraction | $\times 2$ | integer part |
| - | - | - | - |
| 1 | $0.98746$ | $1.97492$ | $\mathbf{1}$ |
| 2 | $0.97492$ | $1.94984$ | $\mathbf{1}$ |
| 3 | $0.94984$ | $1.89968$ | $\mathbf{1}$ |
| 4 | $0.89968$ | $1.79936$ | $\mathbf{1}$ |
| 5 | $0.79936$ | $1.59872$ | $\mathbf{1}$ |
| 6 | $0.59872$ | $1.19744$ | $\mathbf{1}$ |
| 7 | $0.19744$ | $0.39488$ | $\mathbf{0}$ |
| 8 | $0.39488$ | $0.78976$ | $\mathbf{0}$ |
| 9 | $0.78976$ | $1.57952$ | $\mathbf{1}$ |
| 10 | $0.57952$ | $1.15904$ | $\mathbf{1}$ |

Bits 1-10: $\mathbf{1111110011}$

### Worked example 1 - Step 3: fractional part (bits 11-20)

| bit | fraction | $\times 2$ | integer part |
| - | - | - | - |
| 11 | $0.15904$ | $0.31808$ | $\mathbf{0}$ |
| 12 | $0.31808$ | $0.63616$ | $\mathbf{0}$ |
| 13 | $0.63616$ | $1.27232$ | $\mathbf{1}$ |
| 14 | $0.27232$ | $0.54464$ | $\mathbf{0}$ |
| 15 | $0.54464$ | $1.08928$ | $\mathbf{1}$ |
| 16 | $0.08928$ | $0.17856$ | $\mathbf{0}$ |
| 17 | $0.17856$ | $0.35712$ | $\mathbf{0}$ |
| 18 | $0.35712$ | $0.71424$ | $\mathbf{0}$ |
| 19 | $0.71424$ | $1.42848$ | $\mathbf{1}$ ← last stored bit |
| 20 | $0.42848$ | $0.85696$ | $\mathbf{0}$ ← rounding bit |

Bits 11-19: $\mathbf{001010001}$, rounding bit: $\mathbf{0}$

$$0.98746 = (0.1111110011001010001\,|\,0\ldots)_{2}$$

### Worked example 1 - Step 4: normalise and round

$$24.98746 = (11000.1111110011001010001\,|\,0\ldots)_{2}$$

Move the radix point 4 places to the left:

$$24.98746 = (1.\underbrace{10001111110011001010001}_{23\text{ bits}}\,|\,0\ldots)_{2} \times 2^{4}$$

- Rounding bit is $0$ → round down (drop the remaining bits).

$$f = 10001111110011001010001$$

`1 | ???????? | 10001111110011001010001`

### Worked example 1 - Step 5: exponent

$$2^{e - 127} = 2^{4} \quad\Rightarrow\quad e = 4 + 127 = 131$$

$$131 = 128 + 2 + 1 = 2^{7} + 2^{1} + 2^{0} = (10000011)_{2}$$

`1 | 10000011 | 10001111110011001010001`

### Worked example 1 - Step 6: assemble the 32 bits

| sign $s$ | exponent $e$ | mantissa $f$ |
| - | - | - |
| `1` | `10000011` | `10001111110011001010001` |

$$-24.98746 \;\rightarrow\; (1\,10000011\,10001111110011001010001)_{2}$$

Let's check:

In [ ]:
import decimal

import numpy as np

x = np.float32(-24.98746)
print(f'bits: {int(x.view(np.uint32)):032b}')
print(f'stored value: {decimal.Decimal(float(x))}')

The computer does not store exactly $-24.98746$, but the closest number single precision can store, $-24.98745918\ldots$ (rounding error).

### Worked example 2: single precision → decimal

Which decimal number is stored in these 32 bits?

$$(10101011110011010001001000110100)_{2}$$

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-127}$$

1. Split the bits into sign, exponent, and mantissa
2. Sign $s$ and exponent $e$
3. Mantissa $1.f$
4. Put it together

### Worked example 2 - Step 1: split the bits

Single precision: $1 + 8 + 23 = 32$ bits

$$(10101011110011010001001000110100)_{2}$$

| sign $s$ (1 bit) | exponent $e$ (8 bits) | mantissa $f$ (23 bits) |
| - | - | - |
| `1` | `01010111` | `10011010001001000110100` |

### Worked example 2 - Step 2: sign and exponent

- Sign: $s = 1 \;\Rightarrow\; (-1)^{1} = -1$ - the number is negative.
- Exponent:

$$e = (01010111)_{2} = 2^{6} + 2^{4} + 2^{2} + 2^{1} + 2^{0} = 64 + 16 + 4 + 2 + 1 = 87$$

$$e - 127 = 87 - 127 = -40$$

So far: $x = -(1.f)_{2} \times 2^{-40}$

### Worked example 2 - Step 3: mantissa

$$1.f = (1.10011010001001000110100)_{2}$$

Bits equal to $1$ are at positions $1, 4, 5, 7, 11, 14, 18, 19, 21$ after the radix point:

$$1.f = 1 + 2^{-1} + 2^{-4} + 2^{-5} + 2^{-7} + 2^{-11} + 2^{-14} + 2^{-18} + 2^{-19} + 2^{-21}$$

$$1.f = 1.602118015289306640625$$

So far: $x = -1.602118015289306640625 \times 2^{-40}$

### Worked example 2 - Step 4: put it together

$$x = (-1)^{1} \times 1.602118015289306640625 \times 2^{87 - 127}$$

$$x = -1.602118015289306640625 \times 2^{-40} \approx -1.4571178 \times 10^{-12}$$

Let's check:

In [ ]:
bits = '10101011110011010001001000110100'

s, e, m = 1, 87, 1.602118015289306640625
print('by hand:', (-1)**s * m * 2**(e - 127))
print('numpy:  ', np.uint32(int(bits, 2)).view(np.float32))

### Machine $\epsilon$

- The number of real numbers we can represent exactly is finite - we call those numbers **machine numbers**.
- The floating-point machine number corresponding to $x$, we denote as $\mathrm{fl}(x)$.
- Machine $\epsilon$ is the gap between $1$ and the next larger machine number.

- For single precision, the machine $\epsilon$ is:

$$\epsilon = \overbrace{(1.00000000000000000000001)_{2}}^{1+\epsilon} - \overbrace{(1.00000000000000000000000)_{2}}^{1} = \overbrace{2^{-23}}^{\epsilon} \approx 1.19 \times 10^{-7}$$

### Machine numbers are not equally spaced

<center>
    <img src="images/machine-numbers.png" alt="Machine numbers on the number line" width="1300"/>
</center>

### Precision and resolution

- The mantissa has $23$ stored bits plus the hidden $1$ - that is $24$ significant bits.
- The **precision** is $p = \lfloor -\log_{10}(\epsilon) \rfloor = \lfloor -\log_{10}(2^{-23}) \rfloor = \lfloor 6.92 \rfloor = 6$ - we can rely on 6 significant digits.

$$x = \mathbf{5.31245}\overbrace{79873459782349658}^{\text{rubbish}}$$

- **Resolution:** Sometimes, instead of precision $p$, we show **resolution**: $10^{-p} = 10^{-6}$.

### We do not have to memorise that information...

In [ ]:
import numpy as np

single = np.finfo(np.float32)

print('Single precision float')
print(25*'-')
print(f'Total size in bits: {single.bits}')
print(f'Bits in mantissa: {single.nmant}')
print(f'Bits in exponent: {single.nexp}')
print(f'Largest number: {single.max}')
print(f'Smallest number: {single.min}')
print(f'Smallest positive normal number: {single.tiny}')
print(f'Smallest positive subnormal number: {single.smallest_subnormal}')
print(f'Machine epsilon: {single.eps}')
print(f'Precision: {single.precision}')
print(f'Resolution: {single.resolution}')

### Double precision (64 bit)

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-1023}$$

- 1 bit for the sign $s$
    
- 11 bits for the exponent $e$
    - $0 < e < (11\,111\,111\,111)_{2} = 2047$ (values $0$ and $2047$ are reserved for $\pm 0$/subnormals and $\pm \infty$/NaN)
    - $-1022 \le e - 1023 \le 1023$
 
- 52 bits for mantissa $f$
    $$(1.00\ldots 00)_{2} = 1 \le (1.f)_{2} \le (1.11\ldots 11)_{2} = 2 - 2^{-52}$$

- Largest number: $(2 - 2^{-52}) \times 2^{1023} \approx 2^{1024} \approx 1.8 \times 10^{308}$
- Smallest positive normal number: $1 \times 2^{-1022} \approx 2.2 \times 10^{-308}$ (subnormals go down to $2^{-1074} \approx 4.9 \times 10^{-324}$)

### Double precision: machine $\epsilon$, precision, and resolution

- Machine $\epsilon$:

$$\epsilon = 2^{-52} \approx 2.22 \times 10^{-16}$$

- Precision: $p = \lfloor -\log_{10}(2^{-52}) \rfloor = \lfloor 15.65 \rfloor = 15$ significant digits ($53$ significant bits including the hidden $1$).

$$x = \mathbf{5.31245775431986}\overbrace{[9873459782349658]}^{\text{rubbish}}$$

- Resolution: $10^{-p} = 10^{-15}$.

### Do not memorise....

In [ ]:
double = np.finfo(np.float64)

print('Double precision float')
print(25*'-')
print(f'Total size in bits: {double.bits}')
print(f'Bits in mantissa: {double.nmant}')
print(f'Bits in exponent: {double.nexp}')
print(f'Machine epsilon: {double.eps}')
print(f'Largest number: {double.max}')
print(f'Smallest number: {double.min}')
print(f'Smallest positive normal number: {double.tiny}')
print(f'Smallest positive subnormal number: {double.smallest_subnormal}')
print(f'Precision: {double.precision}')
print(f'Resolution: {double.resolution}')

### Overflow and underflow

- How can we represent $x = 2^{68426539}$ or $x = 2^{-98723640}$?
    -  We can't. This attempt results in overflow or underflow.

- **Underflow** is often rounded to zero.
- **Overflow**: the IEEE-754 default result is $\pm\infty$ (`inf` in Python, e.g. `1e308 * 10`). Some Python operations raise `OverflowError` instead (e.g. `2.0**10000`).

In [ ]:
huuuuuuge_number = 1e68426539
super_super_small_number = 1e-98723640

print(f'{huuuuuuge_number = }')
print(f'{super_super_small_number = }')

## Rounding

Let's say we need to compute $a + b$. Since only machine numbers can be represented exactly, the result is

$$\text{fl}(\,\text{fl}(a) + \text{fl}(b)\,)$$

<center>
    <img src="images/rounding.png" alt="Rounding" width="1300"/>
</center>

### Chopping and rounding

- For simplicity, here, we assume single precision floating-point numbers.
- Let us say we want to represent $x$ with its nearest machine number $\mathrm{fl}(x)$:

$$x = (1.b_{1}b_{2}b_{3}b_{4}\ldots)_{2} \times 2^{e}$$

- How large can the error be?

- **Chopping:** We can chop excess bits $b_{24}b_{25}\ldots$ and the machine number is then:

$$x_{-} = (1.b_{1}b_{2}b_{3}\ldots b_{23})_{2} \times 2^{e}$$

- **Rounding:** $x_{+}$ is just to the right and it is obtained by rounding up. We add one unit to $b_{23}$ (the last stored bit).

$$x_{+} = [(1.b_{1}b_{2}b_{3}\ldots b_{23})_{2} + 2^{-23}] \times 2^{e}$$

- We now choose the number which is closer to $x$ as its machine representation $\mathrm{fl}(x)$.

### Absolute and relative error

- **Absolute error**:

$$|x - \mathrm{fl}(x)| \le \frac{1}{2}|x_{+} - x_{-}| = \frac{1}{2} \times 2^{-23} \times 2^{e} = 2^{-24+e}$$

- **Relative error**:

$$\frac{|x - \mathrm{fl}(x)|}{|x|} \le \frac{2^{-24 + e}}{(1.b_{1}b_{2}b_{3}\ldots)_{2} \times 2^{e}} \le \frac{2^{-24}}{1} = 2^{-24} = u$$

- $u$ is the **unit round-off error**.
    - It is $2^{-k}$ where $k$ is the number of bits in mantissa plus the hidden bit (23 + 1 for single precision).

### Floating point addition

1. **Align Exponents**: Adjust the exponents so they are the same, shifting the decimal point as necessary.

- Example: $3.25 \times 10^1$ + $1.75 \times 10^0$
- Shift $1.75 \times 10^0$ to $0.175 \times 10^1$

2. **Add the Mantissas**: Keep the exponent constant, and add the mantissas.

- Example: $3.25 + 0.175 = 3.425$

3. **Normalize the Result**: Adjust the result so that there’s only one non-zero digit before the decimal.

- Example: $3.425 \times 10^1$ is already normalized.

4. **Round the Result (if needed)**: Round to fit within the number of digits supported by the floating-point format.

### Floating point addition: where digits are lost

<center>
    <img src="images/lost-digits-addition.png" alt="Digits lost when adding a small number" width="1300"/>
</center>

### Error in arithmetic operations

- For any $x \in \mathbb{R}$ within the range of the computer:

$$\frac{|x - \mathrm{fl}(x)|}{|x|} \le u$$

$$\mathrm{fl}(x) = x(1 + \delta), \quad |\delta| \le u$$

- If $\odot$ is any operation ($+, -, \times, /$), and we assume that $x \odot y$ is correctly computed, normalised, and finally rounded:

$$\mathrm{fl}(x \odot y) = (x \odot y)(1 + \delta)$$

- In practice, $x$ and $y$ are rounded first, so the computer returns (up to three rounding errors):

$$\mathrm{fl}(\mathrm{fl}(x) \odot \mathrm{fl}(y))$$

## Let's revisit the surprises

### Surprise 1

#### Maths says...
For any $x \in \mathbb{R}$ if $x \ne 0$
$$1 + x \ne 1$$

#### Computer says...

In [ ]:
x = 1e-16

print(1 + x != 1)

### Why?

- $x$ is smaller than half of machine epsilon: $x < \epsilon / 2$
- $1 + x$ is closer to $1$ than to the next machine number $1 + \epsilon$, so it rounds back to $1$.

In [ ]:
double = np.finfo(np.float64)
print(f'{x = }')
print(f'epsilon = {double.eps}')

In [ ]:
x = 3e-16  # now, it's larger than machine epsilon

print(1 + x != 1)

### Surprise 2

#### Maths says...
$$\frac{1}{10} + \frac{2}{10} = \frac{3}{10}$$

$$0.1 + 0.2 = 0.3$$

#### Computer says...

In [ ]:
print(0.1 + 0.2 == 0.3)

### Why?

- We round to the closest machine number:

$$\text{fl}(\,\text{fl}(a) + \text{fl}(b)\,)$$

In [ ]:
import decimal

print('fl(0.1) = ', decimal.Decimal(0.1))
print('fl(0.2) = ', decimal.Decimal(0.2))
print('fl(0.3) = ', decimal.Decimal(0.3))

If a number has a finite representation in decimal form, it does not mean it has a finite representation in binary form.
- $0.1$ in binary form is like $0.3333333...$ in decimal.

$$(0.1)_{10} = (0.00011001100110011001100110011001...)_{2}$$

### Surprise 3

#### Maths says...
$$(a + b) + c = a + (b + c)$$

#### Computer says...

In [ ]:
a = 1e16
b = -1e16
c = 1.0

print('Associativity works:', (a + b) + c == a + (b + c))

In [ ]:
print("(a + b) + c = ", (a + b) + c)
print("a + (b + c) = ", a + (b + c))

### Why?

- We round to the closest machine number:

$$\text{fl}(\,\text{fl}(a) + \text{fl}(b)\,)$$

In [ ]:
print('fl(a) = ', decimal.Decimal(a))
print('fl(b) = ', decimal.Decimal(b))
print('fl(c) = ', decimal.Decimal(c))
print('fl(a + b) = ', decimal.Decimal(a + b))
print('fl(b + c) = ', decimal.Decimal(b + c))

Machine numbers are not equally spaced: near $10^{16}$, neighbouring doubles are $2$ apart, so adding $1$ is rounded away (see the slide "Machine numbers are not equally spaced").

### Surprise 4

#### Maths says...
$$1.0000001 - 1 = 0.0000001 = 10^{-7}$$

#### Computer says...

In [ ]:
a = 1.0000001
b = 1.0000000

print("a - b == 1e-7:", a - b == 1e-7)

In [ ]:
print("a - b =", a - b)

### Why? Subtracting nearly equal numbers

<center>
    <img src="images/lost-digits-subtraction.png" alt="Cancellation of leading digits" width="1300"/>
</center>

### Why? (self-study, talk to ChatGPT/Copilot about it)

**Loss of significance (catastrophic cancellation)**

- $a = 1.0000001$ is not a machine number, so it is stored with a small rounding error:

$$\mathrm{fl}(a) = a(1 + \delta_{a}), \quad |\delta_{a}| \le u \quad (u = 2^{-53} \text{ in double precision})$$

- Subtracting two nearby machine numbers is exact (Sterbenz lemma) - the computer correctly subtracts the numbers it was given.
- The problem: leading digits cancel, and the rounding error of $\mathrm{fl}(a)$ becomes large *relative* to the small result:

$$\frac{|(\mathrm{fl}(a) - \mathrm{fl}(b)) - (a - b)|}{|a - b|} \lesssim \frac{|a| + |b|}{|a - b|}\, u$$

- Here $\frac{|a| + |b|}{|a - b|} \approx 2 \times 10^{7}$, so the relative error of the result can be $\sim 10^{7}$ times larger than $u$.

#### Implications:
1. **Subtraction** of nearly equal values **magnifies relative errors** already present in $a$ and $b$.
2. The more **leading significant digits** of $a$ and $b$ agree, the more digits of the result are lost.
3. If possible, rewrite formulas to avoid subtracting nearly equal numbers.

### Surprise 5

#### Maths says...
$$a \times b = \underbrace{b + b + ... + b}_{\text{a times}}$$

#### Computer says...

In [ ]:
a = 100
b = 0.1

s = 0
for _ in range(a):
    s += b

print("a * b == b + b + b +...:", a * b == s)

In [ ]:
print("a * b =", a * b)
print("b + b + b + b + ...=", s)

### Why?

The error accumulates over many iterations.

In [ ]:
print('fl(0.1) =' , decimal.Decimal(0.1))
print('fl(s) =', decimal.Decimal(s))

## Floating point in practice 🛠️

- Rounding errors are unavoidable: every operation can introduce a relative error up to $u$.
- But we can stop them from **accumulating** and from being **amplified**.

1. Adding many numbers
2. Subtracting nearly equal numbers
3. Huge or tiny intermediate values
4. Stopping and comparing
5. Measuring the error

### Problem 1: adding many numbers

$$\sum_{n=1}^{10^{6}} \frac{1}{n^{2}} \quad \text{(in single precision to make the effect visible)}$$

In [ ]:
import decimal
import math

N = 1_000_000
terms = [np.float32(1 / n**2) for n in range(1, N + 1)]

s = np.float32(0)
for t in terms:  # largest terms first
    s += t

reference = math.fsum(1 / n**2 for n in range(1, N + 1))
print('largest first: ', s)
print(f'reference:      {reference:.10f}')
print(f'relative error: {abs(float(s) - reference) / reference:.1e}')

**Why?**
- Near $s \approx 1.6$, single precision numbers are $\epsilon \approx 1.2 \times 10^{-7}$ apart (machine numbers are not equally spaced).
- Adding $x < \frac{\epsilon}{2}|s|$ does not change $s$ (Surprise 1): terms $1/n^{2} \lesssim 10^{-7}$ ($n \gtrsim 3000$) are lost completely.
- Every `+=` is rounded (Surprise 5): $10^{6}$ small errors accumulate.

### Solution 1a: add small numbers first

- Small terms first add up to a partial sum that is large enough to "survive" when the large terms come.

In [ ]:
s = np.float32(0)
for t in reversed(terms):  # smallest terms first
    s += t

print('smallest first:', s)
print(f'relative error: {abs(float(s) - reference) / reference:.1e}')

**In practice:** If you cannot sort, *pairwise summation* (sum the two halves recursively, as `np.sum` does) keeps partial sums of similar size.

### Solution 1b: compensated (Kahan) summation

- **Idea:** Keep the rounding error of every addition in a separate variable $c$ and add it back in the next step.

In [ ]:
def kahan_sum(values):
    s = 0.0  # running sum
    c = 0.0  # compensation: the part rounding has thrown away
    for x in values:
        y = x - c        # correct the next value by the lost part
        t = s + y        # this addition is rounded...
        c = (t - s) - y  # ...(t - s) is what was really added, minus y gives the rounding error
        s = t
    return s


naive = 0.0
for _ in range(10):
    naive += 0.1

print(f'naive: {naive}')
print(f'Kahan: {kahan_sum([0.1] * 10)}')

**In practice:** `math.fsum` does this (even more accurately). Since Python 3.12, the built-in `sum()` is compensated too, which is why we wrote the loops ourselves.

### Problem 2: subtracting nearly equal numbers

$$\sqrt{x + 1} - \sqrt{x}, \quad x = 10^{12}$$

In [ ]:
x = 1e12
naive = math.sqrt(x + 1) - math.sqrt(x)

with decimal.localcontext() as ctx:
    ctx.prec = 50  # reference with 50 significant digits
    reference = decimal.Decimal(x + 1).sqrt() - decimal.Decimal(x).sqrt()

print(f'naive:     {naive!r}')
print(f'reference: {float(reference)!r}')

**Why?** (Surprise 4, loss of significance)
- $\sqrt{x+1} \approx \sqrt{x} \approx 10^{6}$ are each rounded: absolute error up to $\approx 10^{6} \times u \approx 10^{-10}$.
- They agree in the first ~12 digits, which cancel in the subtraction.
- The rounding errors stay: $\approx 10^{-10}$ on a result of $\approx 5 \times 10^{-7}$. Only ~5 digits are correct.

### Solution 2: reformulate to avoid the subtraction

- Find an algebraically equivalent expression without subtracting close numbers. Here, multiply and divide by the conjugate:

$$\sqrt{x+1} - \sqrt{x} = \frac{(\sqrt{x+1} - \sqrt{x})(\sqrt{x+1} + \sqrt{x})}{\sqrt{x+1} + \sqrt{x}} = \frac{1}{\sqrt{x+1} + \sqrt{x}}$$

- Only additions of positive numbers and a division remain, each with relative error $\le u$.

In [ ]:
better = 1 / (math.sqrt(x + 1) + math.sqrt(x))

print(f'better:    {better!r}')
print(f'reference: {float(reference)!r}')

**In practice:** Libraries provide reformulated functions for common cases. E.g. $\log(1 + x)$ for tiny $x$: `np.log(1 + 1e-10)` gives `1.00000008e-10` ❌, while `np.log1p(1e-10)` gives `9.9999999995e-11` ✅.

### Problem 3: huge or tiny intermediate values

Probability of exactly $1000$ heads in $2000$ coin tosses:

$$P = \binom{2000}{1000} \times 0.5^{2000} \approx 0.0178$$

In [ ]:
n, k = 2000, 1000

binom = 1.0
for i in range(1, k + 1):
    binom *= (n - k + i) / i  # binomial coefficient C(n, k)

print(f'C(n, k) = {binom}')
print(f'0.5**n  = {0.5**n}')
print(f'P       = {binom * 0.5**n}')

**Why?** (overflow and underflow)
- Doubles cover only $\approx 10^{-308} \ldots 10^{308}$.
- $\binom{2000}{1000} \approx 10^{600}$ overflows to `inf`, $0.5^{2000} \approx 10^{-602}$ underflows to `0.0`, and $\infty \times 0$ is `nan`.
- The answer $\approx 10^{-2}$ is perfectly representable. Only the intermediate values are not.

### Solution 3: keep intermediate values moderate

**Idea 1:** Combine large and small factors in every step ($0.5^{2000} = 0.25^{1000}$):

$$P = \prod_{i=1}^{1000} \left(\frac{1000 + i}{i} \times 0.25\right)$$

In [ ]:
P = 1.0
largest = 1.0
for i in range(1, k + 1):
    P *= (n - k + i) / i * 0.25
    largest = max(largest, P)

print(f'P = {P}  (largest intermediate value: {largest:.1e})')

**Idea 2:** Logarithms turn huge products into moderate sums:

$$\log P = \sum_{i=1}^{1000} \left[\log(1000 + i) - \log i\right] + 2000 \log 0.5$$

In [ ]:
log_P = sum(math.log(n - k + i) - math.log(i) for i in range(1, k + 1)) + n * math.log(0.5)

print(f'log P = {log_P:.4f}')
print(f'P = {math.exp(log_P)}')
print(f'exact: {math.comb(n, k) / 2**n}')  # Python integers have unlimited size

Idea 2 is slightly less accurate: an absolute error in $\log P$ becomes a relative error in $P$.

**In practice:** `math.lgamma`, `scipy.special`, and `scipy.stats` work with logarithms internally.

### Problem 4: when to stop, when are two numbers equal?

**Exact comparison** may never become true:

In [ ]:
x = 0.0
while x != 1.0 and x < 1.2:  # without `x < 1.2` this loop never ends
    x += 0.1
    print(x, end='  ')

**Fixed absolute tolerance** does not scale:

In [ ]:
a = 1e20
b = math.nextafter(a, math.inf)  # next machine number after a

print(f'gap between neighbours near 1e20: {b - a}')
print(f'abs(b - a) < 1e-10:               {abs(b - a) < 1e-10}')
print(f'abs(2e-15 - 1e-15) < 1e-10:       {abs(2e-15 - 1e-15) < 1e-10}')

**Why?** Results are rounded (Surprises 2 and 5), so we rarely hit a value exactly. Machine numbers are not equally spaced: the gap between neighbours is $\approx \epsilon |x|$.

### Solution 4: relative tolerance and a maximum number of iterations

- Compare relative to the size of the numbers (like the gap $\approx \epsilon|x|$): $|a - b| \le \text{rtol} \cdot |a|$
- Always limit the number of iterations.

Example: Newton's method for $\sqrt{a}$: $\quad x_{k+1} = \frac{1}{2}\left(x_{k} + \frac{a}{x_{k}}\right)$

In [ ]:
eps = np.finfo(float).eps


def newton_sqrt(a, rtol=4 * eps, max_iter=100):
    x = max(a, 1.0)  # initial guess
    for k in range(1, max_iter + 1):
        x_new = 0.5 * (x + a / x)
        if abs(x_new - x) <= rtol * abs(x_new):  # relative stopping criterion
            return x_new, k
        x = x_new
    raise RuntimeError(f'No convergence after {max_iter} iterations.')


for a in [2.0, 1e40, 1e-40]:
    root, iterations = newton_sqrt(a)
    print(f'a = {a:g}: {root!r} after {iterations} iterations (math.sqrt: {math.sqrt(a)!r})')

**Why a few $\epsilon$?** Once an update is smaller than $\approx \frac{\epsilon}{2}|x|$, it cannot change $x$ any more (Surprise 1). Asking for more is asking for the impossible.

**In practice:** `math.isclose(a, b, rel_tol=...)` or `np.isclose` (see the takehome message).

### Problem 5: measuring the error

- Wrong results look just as convincing as correct ones: `print` shows ~16 digits either way.
- Compare with a reference computed in **higher precision** (e.g. `decimal` with 50 digits).
- Express the error in units of $\epsilon$: a few $\epsilon$ is as good as double precision gets.

In [ ]:
with decimal.localcontext() as ctx:
    ctx.prec = 50
    for a in [2.0, 1e40, 1e-40]:
        root, _ = newton_sqrt(a)
        reference = decimal.Decimal(a).sqrt()
        rel_error = abs((decimal.Decimal(root) - reference) / reference)
        print(f'a = {a:g}: relative error = {float(rel_error) / eps:.2f} eps')

**Also test:** known answers, very large and very small inputs, and how the error grows as the problem gets bigger (more terms, more iterations).

### Floating point in practice: checklist ✅

1. **Adding many numbers?** Add the small ones first, or use compensated (Kahan) summation.
2. **Subtracting nearly equal numbers?** Reformulate the expression algebraically.
3. **Huge or tiny intermediate values?** Combine factors as you go, or work with logarithms.
4. **Stopping or comparing?** Use a relative tolerance plus a maximum number of iterations. Never `==`.
5. **Not sure?** Measure the error against a higher-precision reference.

## Takehome message!!! 🎉🕺

**Do not compare floats using `==`!!!**

Instead, use a **relative** tolerance:

$$|a - b| \le \text{rtol} \cdot \max(|a|, |b|)$$

For convenience, there is `math.isclose` (and `np.isclose` for arrays):

In [ ]:
import math

print(math.isclose(0.1 + 0.2, 0.3))  # relative tolerance (rel_tol=1e-09)
print(np.isclose(0.1 + 0.2, 0.3, rtol=1e-9, atol=0))
print(np.isclose(1e-15, 2e-15))  # careful: np.isclose has atol=1e-08 by default

### EOL Quiz:

Answer 5 MCQs (anonymous): https://forms.office.com/e/4MY5qeDK04

1. For 5 min or so, please think about what the correct answers are and note them down. **Do not submit the form yet.**
2. Discuss your answers with several people around you. Has anyone changed your mind? Now, submit your final answers.

### Feedback form

I’d love to hear your thoughts on today’s lecture — your feedback helps me make future sessions better for you!

https://forms.office.com/e/BiekMgPTMm

# Exercises

### Exercise 1: `dec2bin`

In this exercise, you will write a function to convert a decimal floating point number to its binary representation. Follow these steps to complete the exercise:

1. Create a function `dec2bin_integer(integer_part)`. This function takes an integer part of a decimal number as input and returns its binary representation as a string.


2. Create a function `dec2bin_fractional(fractional_part, max_fractional_digits=52)`. This function should take the fractional part of a decimal number and the maximum number of digits in the fractional part as input. It returns the binary representation of the fractional part as a string. We limit the number of digits in the fractional part to avoid infinite loops - some numbers don't have a finite binary representation.

3. Create a main function `dec2bin(num, max_fractional_digits=52)`. This function should take a decimal float and the maximum number of digits in the fractional part as input. Internally, it separates the integer and fractional parts of the input number and calls `dec2bin_integer` and `dec2bin_fractional` defined above. Finally, it returns the combined binary representation of the integer and fractional parts as a string.

In [26]:
def dec2bin_integer(integer_part):
    raise NotImplementedError

def dec2bin_fractional(fractional_part, max_fractional_digits=52):
    raise NotImplementedError

def dec2bin(num, max_fractional_digits=52):
    raise NotImplementedError

# Example usage.
# assert dec2bin(11.625, max_fractional_digits=10) == '1011.101'

#### Solution

In [27]:
def dec2bin_integer(integer_part):
    """
    Convert the integer part of a decimal number to its binary representation.
    
    Parameters
    ----------
    integer_part : int
        The integer part of the decimal number.
    
    Returns
    -------
    str
        The binary representation of the integer part.

    """
    if integer_part == 0:
        return "0"
    
    integer_binary = ""
    while integer_part > 0:
        integer_binary = str(integer_part % 2) + integer_binary
        integer_part //= 2
    
    return integer_binary

def dec2bin_fractional(fractional_part, max_fractional_digits=52):
    """
    Convert the fractional part of a decimal number to its binary representation.
    
    Parameters
    ----------
    fractional_part : float
        The fractional part of the decimal number.

    max_fractional_digits : int, optional
        The maximum number of digits in the fractional part (default is 52).
    
    Returns
    -------
    str
        The binary representation of the fractional part.

    """
    fractional_binary = []
    while fractional_part and len(fractional_binary) < max_fractional_digits:
        fractional_part *= 2
        bit = int(fractional_part)
        fractional_binary.append(str(bit))
        fractional_part -= bit
    
    return "".join(fractional_binary)

def dec2bin(num, max_fractional_digits=52):
    """
    Convert a decimal float to its binary representation.
    
    Parameters
    ----------
    num : float
        The decimal float to convert.

    max_fractional_digits : int, optional
        The maximum number of digits in the fractional part (default is 52).
    
    Returns
    -------
    str
        The binary representation of the input float.

    """
    if num == 0:
        return "0.0"
    
    # Separate the integer and fractional parts.
    integer_part = int(num)
    fractional_part = num - integer_part
    
    # Convert integer and fractional parts to binary.
    integer_binary = dec2bin_integer(integer_part)
    fractional_binary = dec2bin_fractional(fractional_part, max_fractional_digits)
    
    return f"{integer_binary}.{fractional_binary}"

# Example usage
assert dec2bin(11.625, max_fractional_digits=10) == '1011.101'

### Exercise 2

Let's assume we have a machine (computer) that represents floating-point numbers using the following representation:

$$x = \pm(0.b_{1}b_{2}b_{3})_{2} \times 2^{e}, \quad b_{1}, b_{2}, b_{3}, e \in \{0, 1\}$$

1. How many machine numbers are there? What decimal values have those numbers?
2. How do the machine numbers change if we enforce normalisation (e.g. $b_{1}=1$)?

#### Solution

In [ ]:
import itertools

def compute_decimal(s, b1, b2, b3, e):
    mantissa = b1 * 2**-1 + b2 * 2**-2 + b3 * 2**-3
    return (-1)**s * mantissa * 2**e

machine_numbers = [compute_decimal(*bits) for bits in itertools.product(*[(0, 1) for _ in range(5)])] 
print(f'Machine numbers: {machine_numbers}')

We can see there are duplicates. Let us show and count only unique numbers:

In [ ]:
machine_numbers = set(machine_numbers)
print(f'Machine numbers: {machine_numbers}')
print(f'Number of machine numbers: {len(machine_numbers)}')

In [ ]:
def compute_decimal_normalised(s, b2, b3, e):
    mantissa = 1 * 2**-1 + b2 * 2**-2 + b3 * 2**-3
    return (-1)**s * mantissa * 2**e

machine_numbers = [compute_decimal_normalised(*bits) for bits in itertools.product(*[(0, 1) for _ in range(4)])] 

machine_numbers = set(machine_numbers)
print(f'Machine numbers: {machine_numbers}')
print(f'Number of machine numbers: {len(machine_numbers)}')

### Exercise 3

In Worked example 1, we found the single precision representation of $-24.98746$. What is its **double precision** representation? What are those 64 bits?

$$x = (-1)^{s} \times (1.f)_{2} \times 2^{e-1023}$$

**Hint:** The steps are the same as in Worked example 1, but $f$ has 52 bits and $e$ has 11 bits.

#### Solution

1. **Sign:** $x < 0$, so $s = 1$.

2. **Binary representation:** We now need $52 - 4 = 48$ fractional bits plus the rounding bit - too many to compute by hand, so we use `dec2bin` from Exercise 1.

**Note:** We pass the exact decimal value `Fraction('24.98746')` instead of the float `24.98746`. The float is already rounded to double precision, so its binary expansion would already contain the answer we are trying to find.

In [ ]:
from fractions import Fraction

dec2bin(Fraction('24.98746'), max_fractional_digits=60)

3. **Normalise:** Move the radix point 4 places to the left:

$$(1.\underbrace{1000111111001100101000101101101101100001101110110000}_{52\text{ bits}}\,|\,0\ldots)_{2} \times 2^{4}$$

4. **Round:** The 53rd bit is $0$, so we round down: `f = 1000111111001100101000101101101101100001101110110000`.

5. **Exponent:** $e - 1023 = 4 \;\Rightarrow\; e = 1027$:

In [ ]:
dec2bin_integer(1027)

6. **Assemble:** sign + exponent + mantissa. This is the bit pattern we saw "under the microscope" at the beginning of the IEEE-754 section.

In [ ]:
sign = '1'
exponent = '10000000011'
mantissa = '1000111111001100101000101101101101100001101110110000'

double_precision = sign + exponent + mantissa
print(double_precision)

# Check.
print(double_precision == f'{int(np.float64(-24.98746).view(np.uint64)):064b}')

### Exercise 4

What decimal number corresponds to the following 64 bits in IEEE-754 double precision?

$$(1\,01111010111\,1001101000100100011010000000000000000000000000000000)_{2}$$

**Hint:** The steps are the same as in Worked example 2.

#### Solution

* Sign: the first bit is `1`, so $s = 1$.
* Exponent: the next 11 bits are `01111010111`, so $e = 983$ and $e - 1023 = -40$.
* Mantissa: the remaining 52 bits are `1001101000100100011010000000000000000000000000000000`. Trailing zeros do not change the value:

$$1.f = (1.10011010001001000110100)_{2} = 1.602118015289306640625$$

$$x = (-1)^{1} \times 1.602118015289306640625 \times 2^{-40}$$

This is the same number as in Worked example 2. It can be represented exactly in single precision, so in double precision its mantissa is just padded with zeros.

In [ ]:
s = 1
e = int('01111010111', 2)
m = 1.602118015289306640625

x = (-1)**s * m * 2**(e - 1023)
print(f'{e = }')
print(f'{x = }')

## Further Reading

* IEEE Computer Society. [IEEE Standard for Floating-Point Arithmetic](https://standards.ieee.org/standard/754-2019.html) (2019)
* D. Goldberg. [What every computer scientist should know about floating-point arithmetic](https://doi.org/10.1145/103162.103163). *ACM Computing Surveys* **23**, 5–48. (1991)
* M. L. Overton. [Numerical Computing with IEEE Floating Point Arithmetic](https://doi.org/10.1137/1.9780898718072). *Society for Industrial and Applied Mathematics*. (2001)
* R. Burden, J. Faires, A. M. Burden. Numerical Analysis. Brooks Cole, 10th edition (2015).